In [1]:
pip install google-play-scraper pandas nltk stanza seaborn matplotlib scikit-learn Sastrawi

     ---------------------------------------- 1.6/1.6 MB 5.2 MB/s eta 0:00:00
     ---------------------------------------- 1.7/1.7 MB 9.0 MB/s eta 0:00:00
     -------------------------------------- 294.9/294.9 kB 9.2 MB/s eta 0:00:00
     ---------------------------------------- 8.1/8.1 MB 9.3 MB/s eta 0:00:00
     ---------------------------------------- 8.1/8.1 MB 7.0 MB/s eta 0:00:00
     -------------------------------------- 209.7/209.7 kB 6.4 MB/s eta 0:00:00
     -------------------------------------- 108.3/108.3 kB 6.5 MB/s eta 0:00:00
     -------------------------------------- 309.1/309.1 kB 6.5 MB/s eta 0:00:00
     -------------------------------------- 278.0/278.0 kB 4.2 MB/s eta 0:00:00
     ---------------------------------------- 78.4/78.4 kB 2.2 MB/s eta 0:00:00
     -------------------------------------- 608.4/608.4 kB 7.7 MB/s eta 0:00:00
     -------------------------------------- 437.9/437.9 kB 6.9 MB/s eta 0:00:00
     ---------------------------------------- 64


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# **1. Scraping Review Aplikasi TIX ID**

In [2]:
from google_play_scraper import reviews_all, Sort
import pandas as pd

tix_reviews = reviews_all(
    'id.tix.android',
    lang='id',
    country='id',
    sort=Sort.NEWEST
)

df = pd.DataFrame(tix_reviews)
df = df[['content', 'score', 'at']]
df = df.dropna(subset=['content'])
df == df[df['content'].str.strip() != '']

print(f'Total Review: {len(df)}')
df.head()

df.to_excel('tix_id_reviews.xlsx', index=False)

Total Review: 96284


# **2. Label Sentiment dari Rating**

In [3]:
def label_sentiment(score: int) -> str:
  if score >= 4:
    return 'Positive'
  elif score == 3:
    return 'Neutral'
  else:
    return 'Negative'

df['sentiment'] = df['score'].apply(label_sentiment)
df.head()

,content,score,at,sentiment
0,sangat ramai pengunjung,5,2026-03-23 18:34:22,Positive
1,aplikasi sangat amat tidak berguna. karna buka...,1,2026-03-23 14:20:58,Negative
2,"keren film nya, film yang mendidik",5,2026-03-23 10:14:14,Positive
3,gooooooooood,5,2026-03-23 09:52:56,Positive
4,dengan apikasi digi sangat membantu untuk book...,5,2026-03-23 04:40:21,Positive


# **3. Regex Cleaning**

In [4]:
import re

def clean_text(text: str) -> str:
  text = text.lower()
  text = re.sub(r'http\S+', '', text)
  text = re.sub(r'@\w+', '', text)
  text = re.sub(r'[^a-zA-Z\s]', ' ', text)
  text = re.sub(r'(.)\1{2,}', r'\1', text)
  text = re.sub(r'\s+', ' ', text).strip()
  return text

df['clean_content'] = df['content'].apply(clean_text)

df.head()

,content,score,at,sentiment,clean_content
0,sangat ramai pengunjung,5,2026-03-23 18:34:22,Positive,sangat ramai pengunjung
1,aplikasi sangat amat tidak berguna. karna buka...,1,2026-03-23 14:20:58,Negative,aplikasi sangat amat tidak berguna karna bukan...
2,"keren film nya, film yang mendidik",5,2026-03-23 10:14:14,Positive,keren film nya film yang mendidik
3,gooooooooood,5,2026-03-23 09:52:56,Positive,god
4,dengan apikasi digi sangat membantu untuk book...,5,2026-03-23 04:40:21,Positive,dengan apikasi digi sangat membantu untuk book...


# **4. Stopword Removal**

In [5]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop = stopwords.words('indonesian')

df['no_stopwords'] = df['clean_content'].apply(
    lambda x: ' '.join([word for word in x.split() if word not in stop])
)

df.head()

[nltk_data] Downloading package stopwords to C:\Users\Ratna
[nltk_data]     Amalia\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


,content,score,at,sentiment,clean_content,no_stopwords
0,sangat ramai pengunjung,5,2026-03-23 18:34:22,Positive,sangat ramai pengunjung,ramai pengunjung
1,aplikasi sangat amat tidak berguna. karna buka...,1,2026-03-23 14:20:58,Negative,aplikasi sangat amat tidak berguna karna bukan...,aplikasi berguna karna bukanya cepat beli tike...
2,"keren film nya, film yang mendidik",5,2026-03-23 10:14:14,Positive,keren film nya film yang mendidik,keren film nya film mendidik
3,gooooooooood,5,2026-03-23 09:52:56,Positive,god,god
4,dengan apikasi digi sangat membantu untuk book...,5,2026-03-23 04:40:21,Positive,dengan apikasi digi sangat membantu untuk book...,apikasi digi membantu booking film film


# **5. Tokenization**

In [6]:
def tokenize(text: str):
  return text.split()

df['tokens'] = df['no_stopwords'].apply(tokenize)

df.head()

,content,score,at,sentiment,clean_content,no_stopwords,tokens
0,sangat ramai pengunjung,5,2026-03-23 18:34:22,Positive,sangat ramai pengunjung,ramai pengunjung,"[ramai, pengunjung]"
1,aplikasi sangat amat tidak berguna. karna buka...,1,2026-03-23 14:20:58,Negative,aplikasi sangat amat tidak berguna karna bukan...,aplikasi berguna karna bukanya cepat beli tike...,"[aplikasi, berguna, karna, bukanya, cepat, bel..."
2,"keren film nya, film yang mendidik",5,2026-03-23 10:14:14,Positive,keren film nya film yang mendidik,keren film nya film mendidik,"[keren, film, nya, film, mendidik]"
3,gooooooooood,5,2026-03-23 09:52:56,Positive,god,god,[god]
4,dengan apikasi digi sangat membantu untuk book...,5,2026-03-23 04:40:21,Positive,dengan apikasi digi sangat membantu untuk book...,apikasi digi membantu booking film film,"[apikasi, digi, membantu, booking, film, film]"


# **6. Stemming**

In [7]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stemming(text: str):
  return stemmer.stem(text)

df['stemmed'] = df['no_stopwords'].apply(stemming)

df.head()

,content,score,at,sentiment,clean_content,no_stopwords,tokens,stemmed
0,sangat ramai pengunjung,5,2026-03-23 18:34:22,Positive,sangat ramai pengunjung,ramai pengunjung,"[ramai, pengunjung]",ramai unjung
1,aplikasi sangat amat tidak berguna. karna buka...,1,2026-03-23 14:20:58,Negative,aplikasi sangat amat tidak berguna karna bukan...,aplikasi berguna karna bukanya cepat beli tike...,"[aplikasi, berguna, karna, bukanya, cepat, bel...",aplikasi guna karna buka cepat beli tiket doub...
2,"keren film nya, film yang mendidik",5,2026-03-23 10:14:14,Positive,keren film nya film yang mendidik,keren film nya film mendidik,"[keren, film, nya, film, mendidik]",keren film nya film didik
3,gooooooooood,5,2026-03-23 09:52:56,Positive,god,god,[god],god
4,dengan apikasi digi sangat membantu untuk book...,5,2026-03-23 04:40:21,Positive,dengan apikasi digi sangat membantu untuk book...,apikasi digi membantu booking film film,"[apikasi, digi, membantu, booking, film, film]",apikasi digi bantu booking film film


# **7. POS Tagging dgn Stanza**

In [8]:
import stanza

stanza.download('id')
nlp = stanza.Pipeline('id')

def pos_tagging(text: str):
  doc = nlp(text)
  result = []
  for sentence in doc.sentences:
    for word in sentence.words:
      result.append((word.text, word.upos))
  return result

print(pos_tagging(df['no_stopwords'].iloc[0]))

c:\Coding\pba\PBA-TIXID-SentimentAnalysis\Week-2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-24 19:21:38 INFO: Downloaded file to C:\Users\Ratna Amalia\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\resources.json
2026-03-24 19:21:38 INFO: Downloading default packages for language: id (Indonesian) ...
2026-03-24 19:22:30 INFO: Downloaded file to C:\Users\Ratna Amalia\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\id\default.zip
2026-03-24 19:22:33 INFO: Finished downloading models and saved to C:\Users\Ratna Amalia\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources
2026-03-24 19:22:33 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCE

[('ramai', 'ADJ'), ('pengunjung', 'NOUN')]


# **8. Hitung Distribusi POS**

In [3]:
from collections import Counter

df_pos_sample = df.sample(600, random_state=42)

def get_pos_counts(text: str):
  doc = nlp(text)
  pos_list = []
  for sentence in doc.sentences:
    for word in sentence.words:
      pos_list.append(word.upos)
  return Counter(pos_list)

df_pos_sample['pos_counts'] = df['no_stopwords'].apply(get_pos_counts)
df.head()

NameError: name 'df' is not defined

# **9. Hitung Adjectve**

In [ ]:
def count_adj(text: str) -> int:
  doc = nlp(text)
  count = 0
  for sentence in doc.sentences:
    for word in sentence.words:
      if word.upos == 'ADJ':
        count += 1
  return count

df_pos_sample['adj_count'] = df_pos_sample['no_stopwords'].apply(count_adj)
df.head()

# **10. Visualisasi Rating**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure()
sns.countplot(x='score', data=df)
plt.title('Distribus Rating Aplikasi TIX ID')
plt.show()

# **11. TF-IDF Feature Extraction**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X = vectorizer.fit_transform(df['stemmed'])
y = df['sentiment']

# **12. Bag of Words (BoW)**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

bow_vectorizer = CountVectorizer()

X_bow = bow_vectorizer.fit_transform(df['stemmed'])

print("BoW Shape:":, X_bow.shape)

# **13. N-Gram Feature**

In [ ]:
ngram_vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    max_features=5000
)

X_ngram = ngram_vectorizer.fit_transform(df['stemmed'])

print("N-Gram Shape:", X_ngram.shape)

# **14. Train, Test, Split**

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# **15. Model 1 - Naive Bayes**

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

# **16. Model 2 - Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print ("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

# **17. Model 3 - SVM**

In [ ]:
from sklearn.svm import LinearSVC
svm = LinearSVC()
svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

# **18. Perbandingan Accuracy Model**

In [ ]:
results = {
    "Naive Bayes": accuracy_score(y_test, y_pred_nb),
    "Logistic Regression": accuracy_score(y_test, y_pred_lr),
    "SVM": accuracy_score(y_test, y_pred_svm),
}

print(results)

# **19. Visualisasi Perbandingan Model**

In [ ]:
plt.figure()
plt.bar(results.keys(), results.values())
plt.title("Perbandingan Accuracy Model Sentiment Analysis")
plt.ylabel("Accuracy")
plt.show()